# 02 — Method B: **INT4 PTQ** (TorchAO, 사후 양자화)

> ## ⚙️ 실행 모드 배너 — 이 노트북은 **Azure A100 80GB(japaneast, Spot)에서 실제 실행**됨
> base=**Qwen3-8B**, transformers HF 백엔드, BF16 LoRA + TorchAO int4 tile-packed.
> 학습/평가는 `quantization/v2_run`이 seed 42·43·44로 이미 수행 — 이 노트북은 **실 아티팩트 로드 +
> 1회 생성 데모 + 집계 수치 표시**(수 시간 재학습 없음).

### 이 방법(B) — 무엇/왜/어떻게
- **무엇**: A 머지 모델을 **재학습 없이** 사후 4bit 양자화(TorchAO `Int4WeightOnlyConfig` g128, **TILE_PACKED_TO_4D**).
- **왜**: 양자화의 **비용 하한**(추가 학습 0). 크기 **~2.65× 축소**(**15.27→5.77GB**, `results/three_way_table.json`)하되 품질은 소폭 하락.
- **어떻게**: 임베딩·`lm_head` 제외(tied-weight 안전). int4 저장은 `safe_serialization=False`(torchao 서브클래스).

### 0) 부트스트랩 & 설정 (재현성)

In [1]:
import os, sys, json, glob
here = os.getcwd()
for cand in [here, os.path.dirname(here), os.path.join(here, "pdf_qa_extraction"),
             os.path.dirname(os.path.dirname(here))]:
    if os.path.isdir(os.path.join(cand, "quantization")):
        if cand not in sys.path: sys.path.insert(0, cand)
        os.chdir(cand); break
print("cwd:", os.getcwd())

cwd: /home/azureuser/work/pdf_qa_extraction


In [2]:
from quantization.data_korquad import load_config
import quantization.v2_pipeline as V
cfg = load_config()
SEED = 42  # representative seed for the demo; metrics below aggregate all seeds
BASE = cfg["base_model"]["selected"]
print("base:", BASE, "| seeds:", cfg.get("seeds"), "| eval held-out:", cfg["data"]["eval_size"])

base: Qwen/Qwen3-8B | seeds: [42, 43, 44] | eval held-out: 1000


In [3]:
import platform, torch, transformers, torchao
env = {"python": platform.python_version(), "torch": torch.__version__,
       "transformers": transformers.__version__, "torchao": torchao.__version__,
       "cuda": torch.version.cuda,
       "device": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")}
os.makedirs("quantization/results", exist_ok=True)
json.dump(env, open("quantization/results/env_B.json", "w"), ensure_ascii=False, indent=2)
env

/home/azureuser/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'python': '3.10.12',
 'torch': '2.11.0+cu130',
 'transformers': '4.57.6',
 'torchao': '0.17.0',
 'cuda': '13.0',
 'device': 'NVIDIA A100 80GB PCIe'}

### 1) 산출물 위치 (per-seed 아티팩트 — 학습/양자화는 `v2_run`으로 이미 실행됨)

In [4]:
MDIR = f"quantization/artifacts/B_int4_ptq_seed{SEED}"
print("int4 PTQ:", MDIR, "exists:", os.path.isdir(MDIR))

int4 PTQ: quantization/artifacts/B_int4_ptq_seed42 exists: True


**빌드 로그(실측)** — int4 크기(seed 무관, 포맷 결정적):

In [5]:
for f in sorted(glob.glob("quantization/results/B_build_seed*.json")):
    s = f.split("seed")[-1].split(".")[0]; l = json.load(open(f))
    print(f"seed {s}: size_gb={l['size_gb']}")

seed 42: size_gb=5.7705
seed 43: size_gb=5.7705
seed 44: size_gb=5.7705


### 2) 동작 데모 (필수) — held-out KorQuAD 질문 **1개** 실측 생성

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
data = V.load_slices(cfg); ex = data["eval"][0]
tok = AutoTokenizer.from_pretrained(MDIR)
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = "left"; tok.truncation_side = "left"
model = AutoModelForCausalLM.from_pretrained(MDIR, device_map="cuda")  # int4 dir carries its own config
prompt = V.build_chat_prompt(tok, V.system_prompt(cfg), ex.context, ex.question, None,
                             V.enable_thinking_flag(cfg), True)
enc = tok(prompt, return_tensors="pt", truncation=True, max_length=3072).to(model.device)
out = model.generate(**enc, max_new_tokens=cfg["eval"]["max_new_tokens"], do_sample=False,
                     pad_token_id=tok.pad_token_id)
ans = V.extract_answer(tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True))
print("[문맥]", ex.context[:120], "...")
print("[질문]", ex.question)
print("[정답]", ex.answers)
print("[B_int4_ptq 모델답]", ans)
del model; torch.cuda.empty_cache()

The tokenizer you are loading from 'quantization/artifacts/B_int4_ptq_seed42' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:  50%|█████     | 1/2 [00:01<00:01,  1.01s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.71it/s]

Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.54it/s]


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[문맥] 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도  ...
[질문] 2004년 이명박이 서울시장 재직시절 전면적으로 개선한 것은?
[정답] ['대중교통체계']
[B_int4_ptq 모델답] 대중교통체계


### 3) 수치 — 3 seed 평균±표준편차 (EM/F1/ppl · `results/three_way_table.json`)

In [7]:
tw = json.load(open("quantization/results/three_way_table.json"))
ag = tw["aggregate"]["B_int4_ptq"]
print("B_int4_ptq  base=", ag["base_model"], " seeds=", ag["seeds"], " n_eval=", ag["n_eval"])
print("  F1  = %.3f +/- %.3f" % (ag["f1"]["mean"], ag["f1"]["std"]))
print("  EM  = %.3f +/- %.3f" % (ag["exact_match"]["mean"], ag["exact_match"]["std"]))
print("  ppl = %.3f +/- %.3f" % (ag["perplexity"]["mean"], ag["perplexity"]["std"]))
print("  size_gb =", ag.get("size_gb"), " tok/s(eager) = %.2f" % ag["tok_per_s"]["mean"])

B_int4_ptq  base= Qwen/Qwen3-8B  seeds= [42, 43, 44]  n_eval= 1000
  F1  = 94.190 +/- 0.232
  EM  = 86.400 +/- 0.283
  ppl = 10.094 +/- 0.113
  size_gb = 5.7705  tok/s(eager) = 4.48


### 4) B vs A — *양자화만의* 손실
B는 A 머지의 순수 사후 int4화이므로 A와의 F1 차이가 **양자화 손실**이다 (재학습 0). 이 손실을 C(QAT)가 얼마나 되돌리는지가 트랙의 핵심 → `03_int4_qat.ipynb`.